In [172]:
import pandas as pd



df_train = pd.read_csv('train_data.csv')
df_test = pd.read_csv('test_data.csv')


subtask1 = 'subtask1'
subtask2 = 'subtask2'
subtask3 = 'subtask3'
subtask4 = 'subtask4'
subtask5 = 'subtask5'
"""

# === PRELUCRARE DATE

# === ANTRENARE MODEL
model = ''


# === PREDICTII (Subtask 5)
df_test[subtask5] = model.predict(df_test)

# === CONSTRUIRE FORMAT FINAL CSV
results = []

# Subtask 1–4: cate o valoare unica per subtask
# results.append((1, 1, val)
# results.append((2, 1, val)
# results.append((3, 1, val)
# results.append((4, 1, val)

# Subtask 5: fiecare rand din test
for row in df_test.itertuples():
    results.append((5, row.id, float(row.subtask5)))

df_output = pd.DataFrame(results, columns=['subtaskID', 'datapointID', 'answer'])
df_output.to_csv('final_submission.csv', index=False)


"""

"\n\n# === PRELUCRARE DATE\n\n# === ANTRENARE MODEL\nmodel = ''\n\n\n# === PREDICTII (Subtask 5)\ndf_test[subtask5] = model.predict(df_test)\n\n# === CONSTRUIRE FORMAT FINAL CSV\nresults = []\n\n# Subtask 1–4: cate o valoare unica per subtask\n# results.append((1, 1, val)\n# results.append((2, 1, val)\n# results.append((3, 1, val)\n# results.append((4, 1, val)\n\n# Subtask 5: fiecare rand din test\nfor row in df_test.itertuples():\n    results.append((5, row.id, float(row.subtask5)))\n\ndf_output = pd.DataFrame(results, columns=['subtaskID', 'datapointID', 'answer'])\ndf_output.to_csv('final_submission.csv', index=False)\n\n\n"

In [173]:
import numpy as np
import sklearn.preprocessing as lp

In [174]:
print(df_train.head)

<bound method NDFrame.head of             ID Customer_ID     Month         Name Age          SSN  \
0       0x6762  CUS_0x6fe2   January     Schnurra  25  164-17-6558   
1      0x18b64  CUS_0xb13c      July    Jennifert  33  528-06-6283   
2       0xb8d5  CUS_0xa0c3    August        Chanf  56  355-23-1301   
3       0x2ec3  CUS_0x6732  February       alazsh  50  060-69-0738   
4       0x1b36  CUS_0x8c69   January     Toonkelf  24  070-07-9414   
...        ...         ...       ...          ...  ..          ...   
44202  0x156b4  CUS_0x36f2      July  Phil Wahbag  40  661-06-7195   
44203   0x4f29  CUS_0x664c     April     Lorenzim  19  482-10-4462   
44204  0x18d58  CUS_0x75ba     March        Amani  32  676-38-0279   
44205   0x3441  CUS_0x7047     April      Alexeix  18  560-15-2864   
44206   0xde07  CUS_0x2d84  February      Julienp  23  565-71-0473   

       Occupation Annual_Income  Monthly_Inhand_Salary  Num_Bank_Accounts  \
0      Accountant      62214.36            4997.5300

In [175]:
import re
def to_years(s):
    years = re.search(r'(\d+)\s*year', s, re.I)
    months = re.search(r'(\d+)\s*month', s, re.I)
    y = int(years.group(1)) if years else 0
    m = int(months.group(1)) if months else 0
    return y + m / 12          # use y * 12 + m for total months

In [176]:
matrix = df_train.values



In [177]:
y_train0 = matrix.T[-1]
matrix = np.delete(matrix, [-1], axis=1)
n_train = len(matrix)

In [178]:
matrix = np.vstack((matrix, df_test.values))

In [179]:
print(y_train0[355])

-1


In [180]:
y_train0 = y_train0.astype(int)

y_train = np.eye(3)[y_train0 + 1]
y_train = y_train.astype(float)

In [181]:
cols = matrix.T
cols = np.delete(cols, [0, 1, 3, 5, 13], axis=0)
for i in range(len(cols[17])):
    cols[16][i] =  to_years(cols[16][i])

In [182]:
from numpy._core.umath import NAN

aoh = [0, 2, 13, 17, 20] # fill
ad = np.empty((0, cols.shape[1]))
for i in aoh:
    noh = cols[i]
    ad = np.vstack((ad, (np.eye(np.max(np.unique(noh, return_inverse=True)[1]) + 1)[np.unique(noh, return_inverse=True)[1]].T)))
cols = np.delete(cols, aoh, axis=0)
cols = np.vstack((cols, ad))


cols = np.char.strip(cols.astype(str), "_")
cols[cols == ''] = np.nan
cols = cols.astype(float)

In [183]:
data = cols.T
mask = np.isnan(data).any(axis=1)

data = data[~mask]

In [184]:
import torch
import torch.nn as nn
import tqdm

In [185]:
class mnn(nn.Module):
    def __init__(self, dl, al):
        super().__init__()
        self.nl = len(al)
        self.layer = nn.Sequential()
        for l in range(self.nl):
            self.layer.append(nn.ModuleList())
            self.layer.append(nn.Linear(dl[l], dl[l + 1]))
            self.layer.append(getattr(nn, al[l])())

    def forward(self, x):
        X = x
        for l in range(self.nl):
            Y = self.layer[l][0][X]
            X = self.layer[l][1][Y]
        return X


In [186]:
X_train = data[: n_train]
X_test = data[n_train:]

In [187]:
mean = X_train.mean(axis=0)
std  = X_train.std(axis=0)
std[std == 0] = 1
X_train = (X_train - mean) / std
X_test  = (X_test  - mean) / std
X_train[:, -1] = 1
X_test[:, -1]  = 1

In [188]:
dim_list = [55, 22, 22, 3]
activation_list = ['ReLU', 'ReLU','Identity']
lr = 1e-4
model_credit = mnn(dim_list, activation_list)